In [2]:
# model_name = './tuned_models/cl_tohoku_20230401_040019'

In [1]:
from datetime import datetime
import os


### Choose a model_type
model_type = 'cl_tohoku'
# model_type = 'izumi_lab'


### Set the model name
model_name = ""
if model_type == 'cl_tohoku':
    model_name = 'cl-tohoku/bert-base-japanese-whole-word-masking'
elif model_type == 'izumi_lab':
    model_name = 'izumi-lab/bert-small-japanese'

print(f'{model_name = }')

    
### Set the directory path for output and create the directory
model_save_path = './tuned_models/' + model_type + '_' + datetime.now().strftime("%Y%m%d_%H%M%S")
print(model_save_path)

os.mkdir(model_save_path)

model_name = 'cl-tohoku/bert-base-japanese-whole-word-masking'
./tuned_models/cl_tohoku_20230401_040019


In [3]:
import json
import pandas as pd
from sentence_transformers import InputExample


def json_dataset_to_df(filepath):
    data_dicts_list = list()
    
    with open(filepath) as fp:
        for line_str in fp:
            line_dict = json.loads(line_str)
            data_dicts_list.append(line_dict)
    
    data_df = pd.DataFrame(data_dicts_list)
    
    return data_df


def create_data_training_list(data_train_df):
    data_train_inputexamples_list = list()
    
    for index, row in data_train_df.iterrows():
        sentence1_str = row['sentence1']
        sentence2_str = row['sentence2']
        label_float = float(row['label']) / 5
        data_train_inputexamples_list.append(InputExample(texts=[sentence1_str, sentence2_str], label=label_float))
    
    return data_train_inputexamples_list


def create_data_validation_dict(data_df):
    sentences_1_list = data_df['sentence1'].tolist()
    sentences_2_list = data_df['sentence2'].tolist()
    labels_sr = data_df['label'].astype('float') / 5
    labels_list = labels_sr.tolist()
        
    data_valid_lists_dict = {
        'sentence1': sentences_1_list,
        'sentence2': sentences_2_list,
        'label': labels_list
    }
    return data_valid_lists_dict


In [4]:
# Read dataset files and convert them to the dataframes.
FILEPATH_INPUT_TRAIN = './datasets/jsts-v1.1/train-v1.1.json'
FILEPATH_INPUT_TEST = './datasets/jsts-v1.1/valid-v1.1.json'


data_train_read_df = json_dataset_to_df(FILEPATH_INPUT_TRAIN)
print(data_train_read_df.head())

data_test_df = json_dataset_to_df(FILEPATH_INPUT_TEST)
data_test_lists_dict = create_data_validation_dict(data_test_df)
print(data_test_lists_dict['sentence1'][:3])
print(data_test_lists_dict['sentence2'][:3])
print(data_test_lists_dict['label'][:3])


  sentence_pair_id               yjcaptions_id                sentence1  \
0                0    10005_480798-10996-92616   川べりでサーフボードを持った人たちがいます。   
1                1        100124-104404-104405   二人の男性がジャンボジェット機を見ています。   
2                2        100142-104431-104432       男性が子供を抱き上げて立っています。   
3                3  100142_211843-104431-86282       男性が子供を抱き上げて立っています。   
4                4  100226_429476-104521-40292  ジャンプ台かスキーヤーがジャンプをしています。   

                 sentence2  label  
0    トイレの壁に黒いタオルがかけられています。    0.0  
1     2人の男性が、白い飛行機を眺めています。    3.8  
2     坊主頭の男性が子供を抱いて立っています。    4.0  
3  テーブルを囲みワインの品評会が行われています。    0.2  
4     茶色い猫の顔が、あおむけになっています。    0.0  
['レンガの建物の前を、乳母車を押した女性が歩いています。', '山の上に顔の白い牛が2頭います。', 'バナナを持った人が道路を通行しています。']
['厩舎で馬と女性とが寄り添っています。', '曇り空の山肌で、牛が２匹草を食んでいます。', '道の上をバナナを背負った男性が歩いています。']
[0.0, 0.48, 0.72]


In [ ]:
# Get a pre-trained model
from sentence_transformers import SentenceTransformer, losses, models

transformer = models.Transformer(model_name)

pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling], device='cuda')

# Define your train dataset, the dataloader and the train loss
train_loss = losses.CosineSimilarityLoss(model)

In [ ]:
from sentence_transformers import evaluation
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader


kf = KFold(n_splits=5, shuffle=True)

for index_train, index_valid in kf.split(data_train_read_df):
    data_train_df = data_train_read_df.iloc[index_train]
    data_valid_df = data_train_read_df.iloc[index_valid]
    
    data_train_inputexamples_list = create_data_training_list(data_train_df)
    dataloader_train = DataLoader(
        dataset=data_train_inputexamples_list,
        batch_size=16,  # 1
        shuffle=True,  # False
        sampler=None,
        batch_sampler=None,
        num_workers=0,
        collate_fn=None,
        pin_memory=False,
        drop_last=False,
        timeout=0,
        worker_init_fn=None,
        prefetch_factor=2,
        persistent_workers=False
    )
    
    data_valid_lists_dict = create_data_validation_dict(data_valid_df)
    evaluator = evaluation.EmbeddingSimilarityEvaluator(
        sentences1=data_valid_lists_dict['sentence1'],
        sentences2=data_valid_lists_dict['sentence2'],
        scores=data_valid_lists_dict['label'],
        batch_size=16,
        main_similarity=evaluation.SimilarityFunction.COSINE,
        name='',
        show_progress_bar=True,
        write_csv=True
    )
    model.fit(
        train_objectives=[(dataloader_train, train_loss)],
        evaluator=evaluator,
        epochs=1,
        steps_per_epoch=None,
        scheduler='WarmupLinear',
        warmup_steps=100,  # int: 10000
        # optimizer_class=<class 'torch.optim.adamw.AdamW'>,
        optimizer_params={'lr': 2e-05},
        weight_decay=0.01,
        evaluation_steps=0,
        output_path=model_save_path,
        save_best_model=True,
        max_grad_norm=1,
        use_amp=False,
        callback=None,
        show_progress_bar=True,
        checkpoint_path=None,
        checkpoint_save_steps=500,
        checkpoint_save_total_limit=0
    )
    
    # print(len(data_train_inputexamples_list))
    # print(data_train_inputexamples_list[:5])
    # print(data_valid_df.head())
    # print(data_valid_lists_dict['sentence1'][:3])
    # print(data_valid_lists_dict['sentence2'][:3])
    # print(data_valid_lists_dict['label'][:3])


In [6]:
from sklearn.metrics.pairwise import paired_cosine_distances

# Encode(Embed) sentence data
embeddings1_list = model.encode(data_test_lists_dict['sentence1'])
embeddings2_list = model.encode(data_test_lists_dict['sentence2'])

cossims_list = 1 - (paired_cosine_distances(embeddings1_list, embeddings2_list))

print(cossims_list)

[0.4085955  0.6137537  0.74925184 ... 0.70567304 0.5764108  0.95926225]


In [7]:
from scipy.stats import pearsonr, spearmanr

score_pearson_float, _ = pearsonr(data_test_lists_dict['label'], cossims_list)
score_spearman_float, _ = spearmanr(data_test_lists_dict['label'], cossims_list)

print(f'{score_pearson_float = }')
print(f'{score_spearman_float = }')

score_pearson_float = 0.8797983013477866
score_spearman_float = 0.8350105828169487


In [ ]:
### cl-tohoku
## 5-Fold, epoch:1
# score_pearson_float = 0.8797983013477866
# score_spearman_float = 0.8350105828169487


### izumi-lab
## Before training
# score_pearson_float = 0.6404931475426449
# score_spearman_float = 0.6329425227142002

## epoch:1
# score_pearson_float = 0.8112839175373776
# score_spearman_float = 0.7570156536248307

## 5-Fold, epoch:1
# score_pearson_float = 0.8421832121810837
# score_spearman_float = 0.7870142697712181
